# AWS Agent Registry Push Sync Lambda 배포

MCP 서버가 제공하는 도구는 시간이 지나면서 새로 추가되거나 기존 도구가 업데이트 또는 삭제될 수 있습니다. AWS Agent Registry는 이러한 변경 사항과 동기화된 상태를 유지해야 합니다. 

이 Notebook에서는 자격 증명을 Registry와 공유하지 않고도 runtime 업데이트를 감지하여 Registry를 최신 상태로 유지하는 Lambda 기반 자동 push 동기화 파이프라인을 구성합니다.

## 구성 항목

- MCP 서버용 Registry 및 레코드 생성
- Lambda용 IAM role
- 사용자 지정 service model이 번들로 포함된 Lambda 함수
- `UpdateAgentRuntime` CloudTrail 이벤트에 대해 트리거되는 EventBridge 규칙
- 선택 사항: 계정 B의 cross-account 이벤트 전달

## 사전 요구 사항

- Lambda, IAM role 및 EventBridge 규칙 생성 권한이 있는 IAM 자격 증명
- boto3 설치

## 아키텍처

![아키텍처 다이어그램](architecture.png)

이 다이어그램은 계정 B의 AgentCore Runtime 업데이트가 EventBridge를 통해 계정 A로 전달되고, Lambda 함수가 MCP 서버에서 도구를 조회한 후 일치하는 Registry 레코드를 업데이트하는 전체 이벤트 흐름을 보여 줍니다. 단일 계정 배포에서는 cross-account 전달 단계를 건너뛰고 이벤트가 계정 A 내에서 직접 흐릅니다.

---

## 0. 종속성 설치

`requirements.txt`에서 필수 Python 패키지(`boto3`, `botocore`)를 설치합니다.
이 패키지는 Notebook 전체에서 AWS API를 호출하는 데 필요합니다.

In [ ]:
!pip install -r requirements.txt --quiet --no-warn-conflicts

## 1. 구성

배포의 핵심 파라미터를 설정합니다. 여기에는 AWS 리전, Lambda 함수 이름,
각 MCP 서버 계정에 대한 AgentCore Identity credential provider 매핑이 포함됩니다.
Registry는 섹션 2에서, MCP 서버 레코드는 섹션 3에서 생성합니다.
여러 계정에 MCP 서버가 있는 경우 각 계정의 항목을 `ACCOUNT_CONFIGS`에 추가합니다.
Cross-account 구성에서는 계정 ID를 `CROSS_ACCOUNT_IDS`에 나열합니다.

### 사전 요구 사항

- AgentCore 서비스를 지원하는 boto3 >= 1.42.87
- 다음 권한이 있는 IAM 사용자 또는 role(`ACCOUNT_ID`와 리전은 필요에 따라 변경)
- 이 role에는 Lambda 함수, IAM role 및 EventBridge 규칙을 생성하고 관리할 권한도 필요합니다.

<details>
<summary>Agent Registry에 필요한 IAM policy(클릭하여 펼치기)</summary>

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowCreateRegistry",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:CreateRegistry"],
            "Resource": ["arn:aws:bedrock-agentcore:us-west-2:ACCOUNT_ID:*"]
        },
        {
            "Sid": "AllowGetUpdateDeleteRegistry",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistry",
                "bedrock-agentcore:DeleteRegistry"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:us-west-2:ACCOUNT_ID:registry/*"]
        },
        {
            "Sid": "AllowCreateAndListRecords",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateRegistryRecord",
                "bedrock-agentcore:SearchRegistryRecords"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:us-west-2:ACCOUNT_ID:registry/*"]
        },
        {
            "Sid": "AllowRecordOperations",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistryRecord",
                "bedrock-agentcore:DeleteRegistryRecord",
                "bedrock-agentcore:SubmitRegistryRecordForApproval",
                "bedrock-agentcore:UpdateRegistryRecordStatus"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:us-west-2:ACCOUNT_ID:registry/*/record/*"]
        }
    ]
}
```

</details>

---

In [ ]:
import boto3
import json
import time
import os
import zipfile

# ── 다음 값을 수정하세요 ──────────────────────────────────────────
AWS_REGION = "us-west-2"  # 모든 resource에 사용할 AWS region
LAMBDA_NAME = "registry-push-sync-lambda"  # Lambda function 이름

# Registry 및 MCP 서버 구성
REGISTRY_NAME = "<your-registry-name>"  # 생성할 registry 이름
REGISTRY_ID = None  # Registry를 생성한 뒤 1b절에서 설정됨
MCP_SERVER_NAME = "<mcp-server-name>"  # MCP server registry record 이름
MCP_SERVER_DESCRIPTION = "<description>"  # MCP server 설명
MCP_RUNTIME_ARN = (
    "arn:aws:bedrock-agentcore:<region>:<account-id>:runtime/<runtime-id>"  # MCP server의 Runtime ARN
)

# MCP 서버 계정별 AgentCore Identity credential provider입니다.
# 각 키는 AgentCore Runtime에서 MCP 서버를 호스팅하는 AWS 계정 ID입니다.
# provider_name은 섹션 2에서 생성한 credential provider와 일치해야 합니다.
# scope는 MCP 서버에 필요한 OAuth scope입니다(선택 사항).
ACCOUNT_CONFIGS = {
    "<account-a-id>": {  # Account A(registry 계정)
        "provider_name": "cognito-provider-AcctA",
        "scope": "<resource-server>/access",
    },
    "<account-b-id>": {  # Account B(cross-account MCP server 계정)
        "provider_name": "cognito-provider-AcctB",
        "scope": "<resource-server>/access",
    },
}

# Cross-account: 계정 A로 EventBridge 이벤트를 전달할 수 있는 계정 ID입니다.
# 단일 계정 배포에서는 빈 목록 []으로 둡니다.
CROSS_ACCOUNT_IDS = ["<account-b-id>"]
# ──────────────────────────────────────────────────────────────────

# 계정 A의 boto3 클라이언트 초기화
session = boto3.Session(region_name=AWS_REGION)
iam = session.client("iam")
lambda_client = session.client("lambda")
events_client = session.client("events")
sts = session.client("sts")

# 호출자 identity에서 현재 계정 ID 확인
ACCOUNT_ID = sts.get_caller_identity()["Account"]
print(f"Account: {ACCOUNT_ID} | Region: {AWS_REGION}")

---
<br>

## 2. Registry 및 MCP 서버 레코드 생성

MCP 서버 레코드를 저장할 AWS Agent Registry를 생성합니다. 응답에서 `REGISTRY_ID`를 가져와
이후의 모든 셀에서 사용합니다. Registry가 이미 존재하는 경우 이 셀을 실행하는 대신
섹션 1에서 `REGISTRY_ID`를 직접 설정할 수 있습니다.


In [ ]:
# bedrock-agentcore-control 클라이언트로 Registry 생성
registry_cp = session.client("bedrock-agentcore-control", region_name=AWS_REGION)

try:
    reg_resp = registry_cp.create_registry(
        name=REGISTRY_NAME,
        description=f"Agent Registry for push sync — {REGISTRY_NAME}",
    )
    REGISTRY_ID = reg_resp.get("registryId") or reg_resp.get("registryArn", "").split("/")[-1]
    print(f"Created registry: {REGISTRY_NAME} → ID: {REGISTRY_ID}")
except Exception as e:
    if "already exists" in str(e).lower() or "conflict" in str(e).lower():
        # Registry가 존재하면 Registry 목록에서 ID 검색
        regs = registry_cp.list_registries()
        for r in regs.get("registries", []):
            if r.get("name") == REGISTRY_NAME:
                REGISTRY_ID = r.get("registryId") or r.get("registryArn", "").split("/")[-1]
                break
        print(f"Registry already exists: {REGISTRY_NAME} → ID: {REGISTRY_ID}")
    else:
        raise

if not REGISTRY_ID:
    raise ValueError("Failed to create or find registry. Set REGISTRY_ID manually in section 1.")

# Registry가 READY 상태가 될 때까지 대기
print(f"Waiting for registry {REGISTRY_ID} to become READY...")
for _ in range(12):
    reg_status = registry_cp.get_registry(registryId=REGISTRY_ID).get("status", "")
    if reg_status == "READY":
        break
    time.sleep(5)
print(f"Using REGISTRY_ID: {REGISTRY_ID} (status: {reg_status})")
time.sleep(20)

---
<br>

## 3. MCP 서버용 Registry 레코드 생성

MCP 서버용 Registry 레코드를 생성합니다. Lambda가 동기화 중에 일치하는 레코드를 찾을 수 있도록
레코드의 `server.inlineContent`에 runtime ARN을 입력합니다. 레코드는 DRAFT 상태로 시작하며
Lambda가 업데이트하기 전에 승인을 받아야 합니다.


In [ ]:
# Registry API가 요구하는 형식으로 server schema를 구성합니다.
# websiteUrl에는 인코딩된 runtime ARN이 포함된 MCP 서버 호출 URL이 들어갑니다.
# Lambda는 이 URL을 사용하여 레코드와 MCP 서버를 일치시킵니다.
encoded_arn = MCP_RUNTIME_ARN.replace(":", "%3A").replace("/", "%2F")
mcp_server_url = f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/runtimes/{encoded_arn}/invocations"

server_schema = json.dumps(
    {
        "name": f"io.example/{MCP_SERVER_NAME.lower().replace(' ', '-').replace('_', '-')}",
        "description": MCP_SERVER_DESCRIPTION,
        "version": "1.0.0",
        "title": MCP_SERVER_NAME,
        "websiteUrl": mcp_server_url,
        "packages": [
            {
                "registryType": "pip",
                "identifier": MCP_SERVER_NAME.lower().replace(" ", "-").replace("_", "-"),
                "version": "1.0.0",
                "registryBaseUrl": "https://pypi.org",
                "runtimeHint": "uvx",
                "transport": {"type": "stdio"},
            }
        ],
    }
)

try:
    rec_resp = registry_cp.create_registry_record(
        registryId=REGISTRY_ID,
        name=MCP_SERVER_NAME,
        description=MCP_SERVER_DESCRIPTION,
        descriptorType="MCP",
        recordVersion="1.0",
        descriptors={
            "mcp": {
                "server": {
                    "schemaVersion": "2025-12-11",
                    "inlineContent": server_schema,
                },
            },
        },
    )
    RECORD_ID = (
        rec_resp.get("recordId")
        or rec_resp.get("registryRecordId")
        or rec_resp.get("recordArn", "").split("/")[-1]
        or ""
    )
    print(f"Create response keys: {list(rec_resp.keys())}")
    print(f"Created record: {MCP_SERVER_NAME} → ID: {RECORD_ID} (status: DRAFT)")
except Exception as e:
    if "already exists" in str(e).lower() or "conflict" in str(e).lower():
        # 레코드가 존재하면 레코드 목록에서 ID 검색
        recs = registry_cp.list_registry_records(registryId=REGISTRY_ID)
        for r in recs.get("registryRecords", []):
            if r.get("name") == MCP_SERVER_NAME:
                RECORD_ID = r.get("recordId") or r.get("registryRecordId", "")
                break
        print(f"Record already exists: {MCP_SERVER_NAME} → ID: {RECORD_ID}")
    else:
        raise

print(f"Using RECORD_ID: {RECORD_ID}")
print("Note: Record is in DRAFT status. Run section 1d to approve it.")

<br>

### 3.1. Registry 레코드 승인

레코드를 승인 워크플로에 따라 이동합니다: DRAFT → PENDING_APPROVAL → APPROVED.
Lambda는 APPROVED 레코드에만 도구를 동기화하므로 push 동기화 파이프라인을
사용하기 전에 이 단계가 필요합니다.

In [ ]:
# 승인 요청 제출(DRAFT → PENDING_APPROVAL)
record = registry_cp.get_registry_record(registryId=REGISTRY_ID, recordId=RECORD_ID)
current_status = record.get("status", "UNKNOWN")
print(f"Record {RECORD_ID} ({record.get('name', '?')}) — status: {current_status}")

if current_status == "DRAFT":
    registry_cp.submit_registry_record_for_approval(
        registryId=REGISTRY_ID,
        recordId=RECORD_ID,
    )
    print("Submitted for approval (DRAFT → PENDING_APPROVAL)")
    time.sleep(3)  # 상태가 전환될 때까지 대기

# 승인(PENDING_APPROVAL → APPROVED)
record = registry_cp.get_registry_record(registryId=REGISTRY_ID, recordId=RECORD_ID)
current_status = record.get("status", "UNKNOWN")

if current_status == "PENDING_APPROVAL":
    registry_cp.update_registry_record_status(
        registryId=REGISTRY_ID,
        recordId=RECORD_ID,
        status="APPROVED",
        statusReason="Approved via deployment notebook",
    )
    print("Approved (PENDING_APPROVAL → APPROVED)")
elif current_status == "APPROVED":
    print("Already APPROVED — nothing to do")
else:
    print(f"Unexpected status: {current_status}")

---
<br>

## 4. AgentCore Identity Credential Provider 생성

AgentCore Identity에 workload identity와 OAuth2 credential provider를 생성합니다.
Workload identity는 이 Lambda를 신뢰할 수 있는 호출자로 나타냅니다. Credential
provider가 Cognito client 자격 증명을 안전하게 저장하므로 Lambda의 환경 변수에
client secret을 넣을 필요가 없습니다.

이 단계는 한 번만 실행하세요. 리소스가 이미 존재하면 셀에서 해당 리소스를 건너뜁니다.

In [ ]:
# ── 계정별로 다음 값을 수정하세요 ─────────────────────────────────
# 이 Lambda의 workload identity 이름
WORKLOAD_IDENTITY_NAME = "registry-push-sync-agent"

# 각 항목은 provider 이름을 Cognito OAuth 구성에 매핑합니다.
# Provider 이름은 위의 ACCOUNT_CONFIGS에 설정한 값과 일치해야 합니다.
CREDENTIAL_PROVIDERS = {
    "cognito-provider-AcctA": {
        "token_endpoint": "https://<cognito-domain-a>.auth.<region>.amazoncognito.com/oauth2/token",
        "authorization_endpoint": "https://<cognito-domain-a>.auth.<region>.amazoncognito.com/oauth2/authorize",
        "issuer": "https://cognito-idp.<region>.amazonaws.com/<pool-id-a>",
        "client_id": "<client-id-a>",
        "client_secret": "<client-secret-a>",
    },
    "cognito-provider-AcctB": {
        "token_endpoint": "https://<cognito-domain-b>.auth.<region>.amazoncognito.com/oauth2/token",
        "authorization_endpoint": "https://<cognito-domain-b>.auth.<region>.amazoncognito.com/oauth2/authorize",
        "issuer": "https://cognito-idp.<region>.amazonaws.com/<pool-id-b>",
        "client_id": "<client-id-b>",
        "client_secret": "<client-secret-b>",
    },
}
# ──────────────────────────────────────────────────────────────────

acps_client = session.client("bedrock-agentcore-control", region_name=AWS_REGION)

# 이 Lambda를 신뢰할 수 있는 호출자로 나타내는 workload identity 생성
try:
    wi_resp = acps_client.create_workload_identity(name=WORKLOAD_IDENTITY_NAME)
    print(f"Created workload identity: {WORKLOAD_IDENTITY_NAME} → {wi_resp.get('workloadIdentityArn', '?')}")
except Exception as e:
    if "already exists" in str(e).lower() or "conflict" in str(e).lower():
        print(f"Workload identity already exists: {WORKLOAD_IDENTITY_NAME}")
    else:
        raise

# Credential provider 생성
for provider_name, config in CREDENTIAL_PROVIDERS.items():
    try:
        resp = acps_client.create_oauth2_credential_provider(
            name=provider_name,
            credentialProviderVendor="CustomOauth2",
            oauth2ProviderConfigInput={
                "customOauth2ProviderConfig": {
                    "oauthDiscovery": {
                        "authorizationServerMetadata": {
                            "issuer": config["issuer"],
                            "authorizationEndpoint": config["authorization_endpoint"],
                            "tokenEndpoint": config["token_endpoint"],
                            "responseTypes": ["token"],
                        }
                    },
                    "clientId": config["client_id"],
                    "clientSecret": config["client_secret"],
                }
            },
        )
        print(f"Created credential provider: {provider_name} → {resp.get('credentialProviderArn', '?')}")
    except Exception as e:
        if "already exists" in str(e).lower() or "conflict" in str(e).lower():
            print(f"Credential provider already exists: {provider_name}")
        else:
            raise

---
<br>

## 5. Lambda 함수용 IAM Role 생성

IAM execution role을 생성합니다. 이 role에는 Lambda 서비스가 role을 수임할 수 있도록 하는 trust policy,
CloudWatch Logs를 위한 AWS 관리형 `AWSLambdaBasicExecutionRole` policy, Registry 액세스,
AgentCore Identity token 조회 및 Secrets Manager 액세스 권한을 부여하는 inline policy가 포함됩니다.

In [ ]:
ROLE_NAME = f"{LAMBDA_NAME}-role"

# Trust policy: Lambda 서비스가 이 role을 수임하도록 허용
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    role = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Role for AWS Agent Registry push sync Lambda",
    )
    ROLE_ARN = role["Role"]["Arn"]
    print(f"Created role: {ROLE_ARN}")
except iam.exceptions.EntityAlreadyExistsException:
    ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"
    print(f"Role already exists: {ROLE_ARN}")

# 기본 Lambda 실행 권한 연결(CloudWatch Logs 권한)
iam.attach_role_policy(
    RoleName=ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)

# Registry, AgentCore Identity 및 Secrets Manager 액세스를 위한 inline policy 추가
iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="RegistryAccess",
    PolicyDocument=json.dumps(
        {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Effect": "Allow",
                    "Action": [
                        "bedrock-agentcore:ListRegistryRecords",  # 일치하는 MCP server를 찾도록 record 목록 조회
                        "bedrock-agentcore:GetRegistryRecord",  # ARN 일치 여부를 확인하도록 전체 record 세부 정보 조회
                        "bedrock-agentcore:UpdateRegistryRecord",  # 변경이 감지되면 tool 업데이트
                        "bedrock-agentcore:GetResourceOauth2Token",  # credential provider에서 OAuth token 가져오기
                        "bedrock-agentcore:GetWorkloadAccessToken",  # Lambda용 workload identity token 가져오기
                        "secretsmanager:GetSecretValue",  # AgentCore Identity가 credential을 읽을 때 필요
                    ],
                    "Resource": "*",
                }
            ],
        }
    ),
)
print("Attached policies")

# Role 전파 대기
print("Waiting 10s for IAM propagation...")
time.sleep(10)

---
<br>

## 6. Lambda 함수 빌드 및 생성

이 섹션은 두 개의 셀로 구성됩니다. 첫 번째 셀은 `handler.py`와 `boto3`, `botocore`,
`requests`를 배포 zip으로 패키징합니다. boto3 >= 1.42.87에는 Registry control plane
service model이 기본 포함되어 있으므로 사용자 지정 model 파일이 필요하지 않습니다.
두 번째 셀은 환경 변수를 구성하고 Lambda 함수를 생성하거나 업데이트합니다.

In [ ]:
# handler.py, boto3/botocore(>= 1.42.87), requests 라이브러리를 포함하는
# 배포 zip을 구성합니다. boto3 1.42.87 이상에는
# bedrock-agentcore-registry-control service model이 기본 포함되어 있으므로
# 사용자 지정 model 파일이 필요하지 않습니다.
ZIP_PATH = "handler.zip"

# Lambda zip에 번들로 포함할 종속성을 임시 디렉터리에 설치
import subprocess
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    subprocess.run(
        [
            "pip",
            "install",
            "boto3>=1.42.87",
            "requests",
            "-t",
            tmpdir,
            "--quiet",
            "--no-warn-conflicts",
        ],
        check=True,
    )
    print("Bundled: boto3, botocore, requests")

    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write("handler.py")
        # 임시 디렉터리의 모든 종속성을 번들로 포함
        for root, dirs, files in os.walk(tmpdir):
            for f in files:
                full_path = os.path.join(root, f)
                arcname = os.path.relpath(full_path, tmpdir)
                zf.write(full_path, arcname)

zip_size = os.path.getsize(ZIP_PATH)
print(f"Built {ZIP_PATH} ({zip_size:,} bytes)")

In [ ]:
# ACCOUNT_CONFIGS에서 Lambda 환경 변수를 구성합니다.
# 여기에는 client secret 없이 provider 이름과 scope만 포함됩니다.
# Lambda는 WORKLOAD_IDENTITY_NAME으로 workload token을 가져온 다음
# CREDENTIAL_PROVIDER_{ACCT}로 AgentCore Identity를 통해 OAuth token을 가져옵니다.
env_vars = {
    "REGISTRY_ID": REGISTRY_ID,
    "WORKLOAD_IDENTITY_NAME": WORKLOAD_IDENTITY_NAME,
}
for acct_id, config in ACCOUNT_CONFIGS.items():
    env_vars[f"CREDENTIAL_PROVIDER_{acct_id}"] = config["provider_name"]
    if config.get("scope"):
        env_vars[f"CREDENTIAL_SCOPE_{acct_id}"] = config["scope"]

# Lambda 함수 생성 또는 업데이트
with open(ZIP_PATH, "rb") as f:
    zip_bytes = f.read()

try:
    resp = lambda_client.create_function(
        FunctionName=LAMBDA_NAME,
        Runtime="python3.12",
        Role=ROLE_ARN,
        Handler="handler.handler",
        Code={"ZipFile": zip_bytes},
        Timeout=30,
        MemorySize=128,
        Environment={"Variables": env_vars},
        Description="Syncs MCP server tools to AWS Agent Registry on runtime updates",
    )
    LAMBDA_ARN = resp["FunctionArn"]
    print(f"Created Lambda: {LAMBDA_ARN}")
except lambda_client.exceptions.ResourceConflictException:
    # 함수가 존재하면 코드 및 구성 업데이트
    lambda_client.update_function_code(
        FunctionName=LAMBDA_NAME,
        ZipFile=zip_bytes,
    )
    time.sleep(5)  # 코드가 업데이트될 때까지 대기
    lambda_client.update_function_configuration(
        FunctionName=LAMBDA_NAME,
        Environment={"Variables": env_vars},
    )
    LAMBDA_ARN = f"arn:aws:lambda:{AWS_REGION}:{ACCOUNT_ID}:function:{LAMBDA_NAME}"
    print(f"Updated existing Lambda: {LAMBDA_ARN}")

---
<br>

## 7. EventBridge 규칙 생성
계정 A에서 `UpdateAgentRuntime` 이벤트와 일치하고 이전 셀에서 생성한 Lambda를 대상으로 하는 규칙을 생성합니다.

In [ ]:
RULE_NAME = f"{LAMBDA_NAME}-trigger"

# AgentCore의 UpdateAgentRuntime CloudTrail 이벤트와 일치시킵니다.
# 이 계정에서 runtime이 생성 또는 업데이트되거나 cross-account EventBridge bus에서
# 전달되면 트리거됩니다.
event_pattern = {
    "source": ["aws.bedrock-agentcore"],
    "detail-type": ["AWS API Call via CloudTrail"],
    "detail": {"eventName": ["UpdateAgentRuntime"]},
}

events_client.put_rule(
    Name=RULE_NAME,
    EventPattern=json.dumps(event_pattern),
    State="ENABLED",
    Description="Triggers push sync Lambda on AgentCore runtime updates",
)
print(f"Created EventBridge rule: {RULE_NAME}")

# Lambda를 대상으로 추가
events_client.put_targets(
    Rule=RULE_NAME,
    Targets=[{"Id": "push-sync-lambda", "Arn": LAMBDA_ARN}],
)
print("Added Lambda target")

# EventBridge의 Lambda 호출 허용
RULE_ARN = f"arn:aws:events:{AWS_REGION}:{ACCOUNT_ID}:rule/{RULE_NAME}"
try:
    lambda_client.add_permission(
        FunctionName=LAMBDA_NAME,
        StatementId="eventbridge-invoke",
        Action="lambda:InvokeFunction",
        Principal="events.amazonaws.com",
        SourceArn=RULE_ARN,
    )
    print("Added Lambda invoke permission for EventBridge")
except lambda_client.exceptions.ResourceConflictException:
    print("Lambda invoke permission already exists")

---
<br>

## 8. Cross-Account 설정(선택 사항)

MCP 서버가 Registry(계정 A)와 다른 계정(계정 B)에 있는 경우 이 섹션을 실행합니다.
계정 B의 `UpdateAgentRuntime` 이벤트가 계정 A의 EventBridge bus로 전달되어
push sync Lambda를 트리거하도록 이벤트 전달을 구성합니다.

### 8.1 계정 A의 Event Bus에 대한 권한을 계정 B에 부여

각 cross-account ID가 `events:PutEvents`를 호출할 수 있도록 계정 A의 기본 EventBridge bus에
resource-based policy를 추가합니다. 이 권한이 없으면 계정 B의 전달 규칙이 이벤트를 전송할 수 없습니다.

In [ ]:
# 각 cross-account ID에 계정 A의 기본 bus로 이벤트를 전송할 권한 부여
for acct_id in CROSS_ACCOUNT_IDS:
    try:
        events_client.put_permission(
            EventBusName="default",
            Action="events:PutEvents",
            Principal=acct_id,
            StatementId=f"AllowAccount{acct_id}",
        )
        print(f"Allowed account {acct_id} to send events to this bus")
    except events_client.exceptions.ResourceAlreadyExistsException:
        print(f"Permission for account {acct_id} already exists")

if not CROSS_ACCOUNT_IDS:
    print("No cross-account IDs configured — skipping")

<br>

### 8.2 계정 B 세션 초기화

계정 B의 AWS CLI profile을 사용하여 별도의 boto3 세션을 생성합니다.
`ACCOUNT_B_PROFILE`을 `~/.aws/config`에 구성된 profile 이름으로 설정합니다.

In [ ]:
# ── 다음 값을 수정하세요 ────────────────────────────────────────
ACCOUNT_B_PROFILE = "<account-b-profile>"  # Account B용 AWS CLI profile
# ──────────────────────────────────────────────────────────────────

session_b = boto3.Session(profile_name=ACCOUNT_B_PROFILE, region_name=AWS_REGION)
iam_b = session_b.client("iam")
events_b = session_b.client("events")
sts_b = session_b.client("sts")

ACCOUNT_B_ID = sts_b.get_caller_identity()["Account"]
print(f"Account B: {ACCOUNT_B_ID} | Profile: {ACCOUNT_B_PROFILE}")

<br>

### 8.3 계정 B에 IAM 전달 Role 생성

EventBridge가 이벤트를 전달하기 위해 수임할 수 있는 IAM role을 계정 B에 생성합니다.
Role의 inline policy는 계정 A의 기본 bus에 대한 `events:PutEvents` 권한을 부여합니다.

In [ ]:
# EventBridge가 이벤트 전달을 위해 수임하는 IAM role을 계정 B에 생성
FORWARD_ROLE_NAME = "EventBridgeForwardRole"

trust_policy_b = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "events.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    role_b = iam_b.create_role(
        RoleName=FORWARD_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy_b),
        Description="Allows EventBridge to forward events to Account A",
    )
    FORWARD_ROLE_ARN = role_b["Role"]["Arn"]
    print(f"Created role: {FORWARD_ROLE_ARN}")
except iam_b.exceptions.EntityAlreadyExistsException:
    FORWARD_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_B_ID}:role/{FORWARD_ROLE_NAME}"
    print(f"Role already exists: {FORWARD_ROLE_ARN}")

# Inline policy 연결: 계정 A의 기본 event bus에 PutEvents 허용
iam_b.put_role_policy(
    RoleName=FORWARD_ROLE_NAME,
    PolicyName="PutEventsToAccountA",
    PolicyDocument=json.dumps(
        {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Effect": "Allow",
                    "Action": "events:PutEvents",
                    "Resource": f"arn:aws:events:{AWS_REGION}:{ACCOUNT_ID}:event-bus/default",
                }
            ],
        }
    ),
)
print("Attached forwarding policy")

# 계정 B에서 IAM role 전파 대기
print("Waiting 10s for IAM propagation...")
time.sleep(10)

<br>

### 8.4 계정 B에 EventBridge 전달 규칙 생성

계정 B에서 `UpdateAgentRuntime` CloudTrail 이벤트와 일치하는 규칙을 생성하고
5.3의 IAM role을 사용하여 계정 A의 기본 event bus로 전달합니다.

In [ ]:
# 계정 B에서 UpdateAgentRuntime 이벤트와 일치하는 전달 규칙 생성
FORWARD_RULE_NAME = "forward-runtime-updates"

events_b.put_rule(
    Name=FORWARD_RULE_NAME,
    EventPattern=json.dumps(
        {
            "source": ["aws.bedrock-agentcore"],
            "detail-type": ["AWS API Call via CloudTrail"],
            "detail": {"eventName": ["UpdateAgentRuntime"]},
        }
    ),
    State="ENABLED",
    Description="Forwards UpdateAgentRuntime events to Account A",
)
print(f"Created forwarding rule: {FORWARD_RULE_NAME}")

# 대상: 전달 role을 사용하는 계정 A의 기본 event bus
events_b.put_targets(
    Rule=FORWARD_RULE_NAME,
    Targets=[
        {
            "Id": "account-a-bus",
            "Arn": f"arn:aws:events:{AWS_REGION}:{ACCOUNT_ID}:event-bus/default",
            "RoleArn": FORWARD_ROLE_ARN,
        }
    ],
)
print("Added Account A event bus as target")

---
<br><br>

## 배포 완료

이제 Registry와 MCP 레코드가 생성되었고 Lambda 함수, IAM role 및 EventBridge 규칙이 모두 배포되었습니다.
동일 계정 또는 cross-account에서 `UpdateAgentRuntime` CloudTrail 이벤트가 발생하면
EventBridge가 Lambda를 트리거합니다. Lambda는 MCP 서버에 연결하여 도구를 검색하고
일치하는 Registry 레코드를 자동으로 업데이트합니다.

**아래의 모든 셀은 선택 사항입니다.** 수동 테스트, 로그 확인, 레코드 승인 및 정리를 위해 제공됩니다.


## 9. Lambda 테스트

이 셀은 합성 CloudTrail 이벤트로 Lambda를 수동 호출합니다.

프로덕션 환경에서는 이 작업이 필요하지 않습니다. Runtime이 업데이트될 때마다 EventBridge가
실제 `UpdateAgentRuntime` CloudTrail 이벤트를 자동으로 전달합니다.
이 이벤트의 `detail.requestParameters` 및 `detail.responseElements` 필드에는
`agentRuntimeId`와 `agentRuntimeArn`이 포함됩니다. Lambda는 이 값을 사용하여
MCP 서버 URL을 구성하고 계정 ID를 확인합니다.

수동 테스트에서는 Lambda가 실제 MCP 서버 endpoint를 호출하여 도구를 가져올 수 있도록
유효한 `TEST_RUNTIME_ID`를 제공해야 합니다. `TEST_ACCOUNT_ID`는 현재 세션의 계정(계정 A)으로
설정됩니다. Cross-account runtime을 테스트하는 경우 계정 B의 ID로 설정하세요.

In [ ]:
# 사용자의 runtime ID로 바꾸세요.
# 계정 ID는 구성된 경우 첫 번째 cross-account ID를 기본값으로 사용하고,
# 그렇지 않으면 현재 세션의 계정(계정 A)을 사용합니다.
TEST_ACCOUNT_ID = CROSS_ACCOUNT_IDS[0] if CROSS_ACCOUNT_IDS else ACCOUNT_ID
TEST_RUNTIME_ID = "<runtime-id>"

test_event = {
    "detail-type": "AWS API Call via CloudTrail",
    "source": "aws.bedrock-agentcore",
    "detail": {
        "eventName": "UpdateAgentRuntime",
        "awsRegion": AWS_REGION,
        "requestParameters": {
            "agentRuntimeId": TEST_RUNTIME_ID,
        },
        "responseElements": {
            "agentRuntimeArn": f"arn:aws:bedrock-agentcore:{AWS_REGION}:{TEST_ACCOUNT_ID}:runtime/{TEST_RUNTIME_ID}",
            "agentRuntimeId": TEST_RUNTIME_ID,
            "status": "UPDATING",
        },
    },
}

response = lambda_client.invoke(
    FunctionName=LAMBDA_NAME,
    Payload=json.dumps(test_event),
)

result = json.loads(response["Payload"].read())
if "FunctionError" in response:
    print(f"ERROR: {json.dumps(result, indent=2)}")
else:
    body = json.loads(result.get("body", "{}"))
    print(f"MCP URL: {body.get('mcp_url', '?')}")
    print(f"Tools found: {body.get('tool_count', 0)}")
    print(f"Tools: {body.get('tools', [])}")
    print(f"Sync result: {body.get('sync', {})}")

---
<br>

## 10. Lambda 로그 확인
Lambda 로그에서 Lambda 호출 세부 정보를 확인할 수 있습니다. 

In [ ]:
# Lambda의 최신 CloudWatch log stream 조회
logs_client = session.client("logs")

log_group = f"/aws/lambda/{LAMBDA_NAME}"

# 가장 최근 호출의 최신 log stream 가져오기
streams = logs_client.describe_log_streams(
    logGroupName=log_group,
    orderBy="LastEventTime",
    descending=True,
    limit=1,
)

if streams["logStreams"]:
    stream_name = streams["logStreams"][0]["logStreamName"]
    events = logs_client.get_log_events(
        logGroupName=log_group,
        logStreamName=stream_name,
        limit=20,
    )
    for e in events["events"]:
        msg = e["message"].strip()
        if msg and not msg.startswith("REPORT") and not msg.startswith("END"):
            print(msg)
else:
    print("No log streams found")

---
<br>

## 11. 정리(선택 사항)

이 Notebook에서 생성한 모든 리소스를 제거합니다.

In [ ]:
# 이전 셀 없이도 정리를 실행할 수 있도록 이름 계산
ROLE_NAME = f"{LAMBDA_NAME}-role"
RULE_NAME = f"{LAMBDA_NAME}-trigger"

# EventBridge 대상 및 규칙 제거
try:
    events_client.remove_targets(Rule=RULE_NAME, Ids=["push-sync-lambda"])
    events_client.delete_rule(Name=RULE_NAME)
    print(f"Deleted EventBridge rule: {RULE_NAME}")
except Exception as e:
    print(f"Rule cleanup: {e}")

# Cross-account 권한 제거
for acct_id in CROSS_ACCOUNT_IDS:
    try:
        events_client.remove_permission(
            EventBusName="default",
            StatementId=f"AllowAccount{acct_id}",
        )
        print(f"Removed permission for account {acct_id}")
    except Exception as e:
        print(f"Permission cleanup: {e}")

# Lambda 삭제
try:
    lambda_client.delete_function(FunctionName=LAMBDA_NAME)
    print(f"Deleted Lambda: {LAMBDA_NAME}")
except Exception as e:
    print(f"Lambda cleanup: {e}")

# IAM role 삭제(먼저 policy 연결 해제)
try:
    iam.detach_role_policy(
        RoleName=ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    )
    iam.delete_role_policy(RoleName=ROLE_NAME, PolicyName="RegistryAccess")
    iam.delete_role(RoleName=ROLE_NAME)
    print(f"Deleted IAM role: {ROLE_NAME}")
except Exception as e:
    print(f"Role cleanup: {e}")

print("Cleanup complete.")

# ── Registry 정리 ────────────────────────────────────────────────
try:
    reg_cleanup = session.client("bedrock-agentcore-control", region_name=AWS_REGION)
    # 레코드가 남아 있으면 Registry를 삭제할 수 없으므로 Registry 레코드부터 삭제
    if "RECORD_ID" in dir() and RECORD_ID:
        try:
            reg_cleanup.delete_registry_record(registryId=REGISTRY_ID, recordId=RECORD_ID)
            print(f"Deleted registry record: {RECORD_ID}")
        except Exception as e:
            print(f"Record cleanup: {e}")
    # Registry 삭제
    if REGISTRY_ID:
        try:
            reg_cleanup.delete_registry(registryId=REGISTRY_ID)
            print(f"Deleted registry: {REGISTRY_ID}")
        except Exception as e:
            print(f"Registry cleanup: {e}")
except Exception as e:
    print(f"Registry cleanup skipped: {e}")

# ── AgentCore Identity 정리 ──────────────────────────────────────
try:
    acps_cleanup = session.client("bedrock-agentcore-control", region_name=AWS_REGION)
    # Credential provider 삭제
    for provider_name in CREDENTIAL_PROVIDERS.keys():
        try:
            acps_cleanup.delete_oauth2_credential_provider(name=provider_name)
            print(f"Deleted credential provider: {provider_name}")
        except Exception as e:
            print(f"Credential provider cleanup ({provider_name}): {e}")
    # Workload identity 삭제
    try:
        acps_cleanup.delete_workload_identity(name=WORKLOAD_IDENTITY_NAME)
        print(f"Deleted workload identity: {WORKLOAD_IDENTITY_NAME}")
    except Exception as e:
        print(f"Workload identity cleanup: {e}")
except Exception as e:
    print(f"AgentCore Identity cleanup skipped: {e}")

# ── 계정 B 정리(cross-account, 선택 사항) ───────────────────────
# ACCOUNT_B_PROFILE이 설정되어 있고 placeholder가 아닐 때만 실행합니다.
if "session_b" in dir() or (ACCOUNT_B_PROFILE and ACCOUNT_B_PROFILE != "<account-b-profile>"):
    try:
        session_b = boto3.Session(profile_name=ACCOUNT_B_PROFILE, region_name=AWS_REGION)
        iam_b = session_b.client("iam")
        events_b = session_b.client("events")
        FORWARD_ROLE_NAME = "EventBridgeForwardRole"
        FORWARD_RULE_NAME = "forward-runtime-updates"
        # 전달 규칙의 대상 및 규칙 제거
        try:
            events_b.remove_targets(Rule=FORWARD_RULE_NAME, Ids=["account-a-bus"])
            events_b.delete_rule(Name=FORWARD_RULE_NAME)
            print(f"Deleted Account B forwarding rule: {FORWARD_RULE_NAME}")
        except Exception as e:
            print(f"Account B rule cleanup: {e}")
        # 전달 IAM role 제거
        try:
            iam_b.delete_role_policy(RoleName=FORWARD_ROLE_NAME, PolicyName="PutEventsToAccountA")
            iam_b.delete_role(RoleName=FORWARD_ROLE_NAME)
            print(f"Deleted Account B role: {FORWARD_ROLE_NAME}")
        except Exception as e:
            print(f"Account B role cleanup: {e}")
    except Exception as e:
        print(f"Account B cleanup skipped: {e}")
else:
    print("Account B cleanup skipped — no profile configured")